# HW2-1 Phoneme Classification
任務：從語音的 MFCC 特徵，逐 frame 預測 Phoneme（音素），共 39 類。

目標：通過 Strong Baseline（Kaggle accuracy > 0.76023）

## 1. 下載資料

In [1]:
# 從 Google Drive 下載 TIMIT 資料集（TA 已預處理）
!gdown --id '1HPkcmQmFGu-3OknddKIa5dNDsR05lIQR' --output data.zip
!unzip data.zip
!ls

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1HPkcmQmFGu-3OknddKIa5dNDsR05lIQR
From (redirected): https://drive.google.com/uc?id=1HPkcmQmFGu-3OknddKIa5dNDsR05lIQR&confirm=t&uuid=353ddd81-e210-462f-95af-7af819db6925
To: /content/data.zip
100% 372M/372M [00:01<00:00, 193MB/s]
Archive:  data.zip
   creating: timit_11/
  inflating: timit_11/train_11.npy   
  inflating: timit_11/test_11.npy    
  inflating: timit_11/train_label_11.npy  
data.zip  sample_data  timit_11


## 2. 載入資料

In [15]:
import numpy as np

print('Loading data ...')

data_root = './timit_11/'

# train_11.npy：shape (N, 429)，每筆是 11 個 frame 的 MFCC 攤平（11 * 39 = 429）
# train_label_11.npy：shape (N,)，label 是 0–38，對應中間那個 frame
# test_11.npy：shape (M, 429)，無 label，需預測
train = np.load(data_root + 'train_11.npy')
train_label = np.load(data_root + 'train_label_11.npy')
test = np.load(data_root + 'test_11.npy')

print('Size of training data: {}'.format(train.shape))
print('Size of testing data: {}'.format(test.shape))

Loading data ...
Size of training data: (1229932, 429)
Size of testing data: (451552, 429)


## 3. 定義 Dataset

In [16]:
import torch
from torch.utils.data import Dataset

class TIMITDataset(Dataset):
    def __init__(self, X, y=None):
        # 將 numpy array 轉成 PyTorch tensor，型別為 float32
        self.data = torch.from_numpy(X).float()
        if y is not None:
            # train_label_11.npy 的 dtype 是 str_，需先轉成 int64
            # 才能建立 LongTensor 傳入 CrossEntropyLoss
            self.label = torch.LongTensor(y.astype(np.int64))
        else:
            self.label = None

    def __getitem__(self, idx):
        if self.label is not None:
            return self.data[idx], self.label[idx]
        else:
            return self.data[idx]

    def __len__(self):
        return len(self.data)

## 4. 切分訓練／驗證集

In [17]:
# 用前 80% 訓練，後 20% 驗證
VAL_RATIO = 0.2

percent = int(train.shape[0] * (1 - VAL_RATIO))
train_x, train_y = train[:percent], train_label[:percent]
val_x, val_y = train[percent:], train_label[percent:]

print('Size of training set: {}'.format(train_x.shape))
print('Size of validation set: {}'.format(val_x.shape))

Size of training set: (983945, 429)
Size of validation set: (245987, 429)


## 5. 建立 DataLoader

In [18]:
from torch.utils.data import DataLoader

# Batch size 調大到 512，讓 Batch Normalization 的統計更穩定，也加速訓練
BATCH_SIZE = 512

train_set = TIMITDataset(train_x, train_y)
val_set = TIMITDataset(val_x, val_y)

# 訓練集 shuffle=True，讓每個 epoch 的 batch 組成不同，有助防止 overfitting
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

## 6. 清理記憶體

In [19]:
import gc

# 釋放已不需要的原始 numpy array，節省 Colab RAM
del train, train_label, train_x, train_y, val_x, val_y
gc.collect()

print('Memory cleaned.')

Memory cleaned.


## 7. 定義模型

相較 Simple Baseline 的改動：
- Sigmoid → **ReLU**（避免梯度消失，訓練更快更穩）
- 加入 **Batch Normalization**（穩定每層輸入分佈，讓訓練更穩定）
- 加入 **Dropout**（隨機丟棄神經元，防止 overfitting）
- 層數：3 層 → **5 層**，增加模型容量

In [20]:
import torch
import torch.nn as nn

class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()

        # 定義各層：Linear → BatchNorm → ReLU → Dropout
        # BatchNorm 放在 ReLU 前，是常見的做法（也有人放後，效果相近）
        self.fc1 = nn.Linear(429, 2048)  #1024->2048. 1-4放大10倍
        self.bn1 = nn.BatchNorm1d(2048)

        self.fc2 = nn.Linear(2048, 2048)#1024->2048
        self.bn2 = nn.BatchNorm1d(2048)

        self.fc3 = nn.Linear(2048, 1024)
        self.bn3 = nn.BatchNorm1d(1024)

        self.fc4 = nn.Linear(1024, 512)
        self.bn4 = nn.BatchNorm1d(512)

        self.fc5 = nn.Linear(512, 128)
        self.bn5 = nn.BatchNorm1d(128)

        # 輸出層：不接 BatchNorm 和 Activation，直接輸出 39 個 logit
        # CrossEntropyLoss 內部會做 Softmax，所以這裡不加
        self.out = nn.Linear(128, 39)

        self.act_fn = nn.ReLU()
        # Dropout p=0.15：每次 forward 隨機丟棄 15% 的神經元
        self.dropout = nn.Dropout(p=0.15)

    def forward(self, x):
        x = self.dropout(self.act_fn(self.bn1(self.fc1(x))))
        x = self.dropout(self.act_fn(self.bn2(self.fc2(x))))
        x = self.dropout(self.act_fn(self.bn3(self.fc3(x))))
        x = self.dropout(self.act_fn(self.bn4(self.fc4(x))))
        x = self.dropout(self.act_fn(self.bn5(self.fc5(x))))
        x = self.out(x)
        return x

## 8. 工具函式

In [21]:
def get_device():
    # 優先使用 GPU（Colab 提供免費 GPU，記得在 Runtime > Change runtime type 選 GPU）
    return 'cuda' if torch.cuda.is_available() else 'cpu'

def same_seeds(seed):
    # 固定所有隨機種子，確保實驗可重現
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

## 9. 初始化訓練參數

In [22]:
same_seeds(0)

device = get_device()
print(f'DEVICE: {device}')

# 訓練 70 個 epoch，搭配 LR Scheduler，讓後期可以更精細地收斂
num_epoch = 100
# 初始 Learning Rate 設 1e-3，比原版大，搭配 Scheduler 自動調降
learning_rate = 1e-3

model_path = './model.ckpt'

model = Classifier().to(device)

# CrossEntropyLoss：多分類標準 Loss，內部包含 Softmax + NLLLoss
criterion = nn.CrossEntropyLoss()

# AdamW：Adam + Weight Decay，相較 Adam 更能防止 overfitting
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

# ReduceLROnPlateau：當 val_loss 連續 patience 個 epoch 沒改善，就把 LR 乘上 factor
# patience=8：等 8 個 epoch 沒進步才降 LR
# factor=0.5：LR 減半
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=8
)

DEVICE: cuda


## 10. 訓練迴圈

In [23]:
best_acc = 0.0

for epoch in range(num_epoch):
    train_acc = 0.0
    train_loss = 0.0
    val_acc = 0.0
    val_loss = 0.0

    # --- 訓練階段 ---
    # model.train() 會啟用 Dropout 和 BatchNorm 的訓練模式
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()          # 清除上一個 batch 的梯度
        outputs = model(inputs)        # Forward pass
        batch_loss = criterion(outputs, labels)
        batch_loss.backward()          # Backward pass，計算梯度
        optimizer.step()               # 更新參數

        _, train_pred = torch.max(outputs, 1)  # 取最大 logit 的 class 作為預測
        train_acc += (train_pred.cpu() == labels.cpu()).sum().item()
        train_loss += batch_loss.item()

    # --- 驗證階段 ---
    # model.eval() 會關閉 Dropout，BatchNorm 改用全局統計
    model.eval()
    with torch.no_grad():  # 驗證時不需要計算梯度，省記憶體和時間
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            batch_loss = criterion(outputs, labels)
            _, val_pred = torch.max(outputs, 1)

            val_acc += (val_pred.cpu() == labels.cpu()).sum().item()
            val_loss += batch_loss.item()

    # 計算當前 epoch 的平均 Loss，傳給 Scheduler 判斷是否需要降 LR
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)

    print('[{:03d}/{:03d}] Train Acc: {:3.6f} Loss: {:3.6f} | Val Acc: {:3.6f} Loss: {:3.6f}'.format(
        epoch + 1, num_epoch,
        train_acc / len(train_set), train_loss / len(train_loader),
        val_acc / len(val_set), avg_val_loss
    ))

    # 儲存在驗證集上表現最好的模型
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), model_path)
        print('  -> saving model with acc {:.3f}'.format(best_acc / len(val_set)))

print('\nTraining complete. Best val acc: {:.3f}'.format(best_acc / len(val_set)))

[001/100] Train Acc: 0.635615 Loss: 1.182097 | Val Acc: 0.700293 Loss: 0.926023
  -> saving model with acc 0.700
[002/100] Train Acc: 0.702335 Loss: 0.923113 | Val Acc: 0.722750 Loss: 0.844992
  -> saving model with acc 0.723
[003/100] Train Acc: 0.730985 Loss: 0.818217 | Val Acc: 0.734258 Loss: 0.808655
  -> saving model with acc 0.734
[004/100] Train Acc: 0.755142 Loss: 0.734708 | Val Acc: 0.737775 Loss: 0.799119
  -> saving model with acc 0.738
[005/100] Train Acc: 0.776101 Loss: 0.661534 | Val Acc: 0.741173 Loss: 0.802402
  -> saving model with acc 0.741
[006/100] Train Acc: 0.796052 Loss: 0.594771 | Val Acc: 0.742637 Loss: 0.816555
  -> saving model with acc 0.743
[007/100] Train Acc: 0.814564 Loss: 0.536467 | Val Acc: 0.740795 Loss: 0.840516
[008/100] Train Acc: 0.829924 Loss: 0.487113 | Val Acc: 0.741246 Loss: 0.868413
[009/100] Train Acc: 0.843634 Loss: 0.443901 | Val Acc: 0.739490 Loss: 0.895670
[010/100] Train Acc: 0.855429 Loss: 0.407329 | Val Acc: 0.738848 Loss: 0.933611
[0

## 11. 預測並輸出結果

In [12]:
# 建立測試集（無 label）
test_set = TIMITDataset(test, None)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

# 載入訓練過程中儲存的最佳模型
model = Classifier().to(device)
model.load_state_dict(torch.load(model_path))

predict = []
model.eval()
with torch.no_grad():
    for inputs in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, test_pred = torch.max(outputs, 1)
        predict.extend(test_pred.cpu().numpy())

# 輸出成 Kaggle 要求的 CSV 格式
with open('prediction.csv', 'w') as f:
    f.write('Id,Class\n')
    for i, y in enumerate(predict):
        f.write('{},{}\n'.format(i, y))

print('prediction.csv saved. Total predictions: {}'.format(len(predict)))

prediction.csv saved. Total predictions: 451552
